# 07 — Aporte de category_id, fechas y precio (CV)

Mide con CV lo que `06_eda_nuevas.ipynb` encontró. Modelo fijo (`HistGradientBoostingClassifier`, mismos hiperparámetros que el mejor Gradient Boosting de `train.ipynb`), misma CV. Bloques acumulativos: Base (lo ya confirmado: B+C+D+E+título+vendedor) → K (`category_id` con `TargetEncoder`) → L (fechas) → M (precio).

Para `category_id` uso `sklearn.preprocessing.TargetEncoder` (cross-fitting + *smoothing*), no el *leave-one-out* de `VendedorEncoder` — con 75% de categorías con 5 publicaciones o menos, el *leave-one-out* sufriría mucho el problema de inversión en grupos chicos. No está en el material (`Tratamiento-de-variables-categóricas.pdf` solo muestra el target encoder "naive", sin este resguardo).

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import warnings

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, TargetEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

from features import aplanar_anidadas, agregar_logs, CapeoPorMoneda, VendedorEncoder

SEED = 42

train = pd.read_json("../data/train_data.jsonlines", lines=True)
y = (train["condition"] == "new").astype(int)
train = aplanar_anidadas(train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

modelo_fijo = HistGradientBoostingClassifier(
    learning_rate=0.1, max_depth=None, max_iter=200, random_state=SEED
)

resultados_bloques = []


def evaluar_bloque(nombre, columnas_categoricas, columnas_numericas, usar_tfidf=False,
                    max_features_tfidf=500, vendedor=None, usar_categoria=False):
    transformers = [
        ("onehot", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),
        ("num", "passthrough", columnas_numericas),
    ]
    if usar_tfidf:
        transformers.append(("tfidf", TfidfVectorizer(max_features=max_features_tfidf), "title"))
    if vendedor == "simple":
        transformers.append(("vendedor", VendedorEncoder(incluir_target=False), ["seller_id", "log_price"]))
    elif vendedor == "target":
        transformers.append(("vendedor", VendedorEncoder(incluir_target=True), ["seller_id", "log_price"]))
    if usar_categoria:
        transformers.append(("categoria", TargetEncoder(target_type="binary", random_state=SEED), ["category_id"]))

    preprocesador = ColumnTransformer(transformers, sparse_threshold=0)
    pipeline = Pipeline([
        ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
        ("logs", FunctionTransformer(agregar_logs)),
        ("preprocesador", preprocesador),
        ("modelo", modelo_fijo),
    ])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        resultados = cross_validate(pipeline, train, y, cv=cv, scoring=["accuracy", "roc_auc"])

    fila = {
        "bloque": nombre,
        "accuracy_media": resultados["test_accuracy"].mean(),
        "accuracy_std": resultados["test_accuracy"].std(),
        "roc_auc_media": resultados["test_roc_auc"].mean(),
    }
    resultados_bloques.append(fila)
    print(nombre, "-> accuracy:", round(fila["accuracy_media"], 4), "+/-", round(fila["accuracy_std"], 4),
          "| roc_auc:", round(fila["roc_auc_media"], 4))
    return fila


columnas_categoricas_base = ["listing_type_id", "buying_mode", "shipping_mode", "seller_state"]
columnas_numericas_base = [
    "log_price", "log_available_quantity", "automatic_relist",
    "sold_quantity", "initial_quantity",
    "n_fotos", "n_atributos", "n_variaciones", "tiene_video", "tiene_garantia",
    "free_shipping", "local_pick_up", "n_metodos_pago", "accepts_mercadopago",
    "es_tienda_oficial", "tiene_descuento", "tiene_oferta", "tiene_parent_item",
]

## Base — B+C+D+E + título + vendedor

Reproduce exactamente lo que ya tiene `train.ipynb`. Si no da ≈0,918, paro y aviso.

In [2]:
evaluar_bloque("Base", columnas_categoricas_base, columnas_numericas_base, usar_tfidf=True, vendedor="target")

Base -> accuracy: 0.9184 +/- 0.0021 | roc_auc: 0.9757


{'bloque': 'Base',
 'accuracy_media': np.float64(0.9184000000000001),
 'accuracy_std': np.float64(0.0020645699260601305),
 'roc_auc_media': np.float64(0.9756692550472585)}

## Bloque K — + category_id (TargetEncoder)

Suma `category_id` codificada con `TargetEncoder(target_type="binary")` — cross-fitting interno + *smoothing* para categorías chicas.

In [3]:
evaluar_bloque("K - + category_id", columnas_categoricas_base, columnas_numericas_base,
               usar_tfidf=True, vendedor="target", usar_categoria=True)

K - + category_id -> accuracy: 0.9233 +/- 0.001 | roc_auc: 0.9783


{'bloque': 'K - + category_id',
 'accuracy_media': np.float64(0.9232857142857143),
 'accuracy_std': np.float64(0.0010010198880774112),
 'roc_auc_media': np.float64(0.9782892855376417)}

## Bloque L — + fechas (hora_inicio, gap_actualizacion)

Calculo las dos columnas que justificó el EDA: `hora_inicio` (hora de `start_time`) y `gap_actualizacion` (`last_updated - start_time`, en días). Son determinísticas (no aprenden nada de los datos), así que se agregan directo a `train`, igual que `aplanar_anidadas`.

In [4]:
train["last_updated"] = pd.to_datetime(train["last_updated"]).dt.tz_localize(None)
train["hora_inicio"] = train["start_time"].dt.hour
train["gap_actualizacion"] = (train["last_updated"] - train["start_time"]).dt.total_seconds() / 86400

columnas_numericas_L = columnas_numericas_base + ["hora_inicio", "gap_actualizacion"]

evaluar_bloque("L - + fechas", columnas_categoricas_base, columnas_numericas_L,
               usar_tfidf=True, vendedor="target", usar_categoria=True)

L - + fechas -> accuracy: 0.9251 +/- 0.0017 | roc_auc: 0.979


{'bloque': 'L - + fechas',
 'accuracy_media': np.float64(0.9251000000000001),
 'accuracy_std': np.float64(0.001728866562762096),
 'roc_auc_media': np.float64(0.9790008315566731)}

## Bloque M — + precio (redondez)

Las 5 flags que confirmó el EDA, calculadas sobre `price` **sin capear** (antes del `Pipeline`) — la redondez del número tal cual lo escribió el vendedor, no después de winsorizar el 0,5% superior.

In [5]:
precio = train["price"]
train["tiene_decimales"] = precio % 1 != 0
train["mult_100"] = precio % 100 == 0
train["mult_1000"] = precio % 1000 == 0
train["termina_9"] = precio.astype("int64") % 10 == 9
train["termina_99"] = precio.astype("int64") % 100 == 99

columnas_numericas_M = columnas_numericas_L + [
    "tiene_decimales", "mult_100", "mult_1000", "termina_9", "termina_99",
]

evaluar_bloque("M - + precio (redondez)", columnas_categoricas_base, columnas_numericas_M,
               usar_tfidf=True, vendedor="target", usar_categoria=True)

M - + precio (redondez) -> accuracy: 0.9267 +/- 0.0021 | roc_auc: 0.9795


{'bloque': 'M - + precio (redondez)',
 'accuracy_media': np.float64(0.9267),
 'accuracy_std': np.float64(0.0020615033151482865),
 'roc_auc_media': np.float64(0.9795071008424351)}

## Tabla final y conclusión

In [6]:
tabla_nuevas = pd.DataFrame(resultados_bloques).set_index("bloque")
tabla_nuevas["ganancia_accuracy"] = tabla_nuevas["accuracy_media"].diff()
tabla_nuevas

,accuracy_media,accuracy_std,roc_auc_media,ganancia_accuracy
bloque,,,,
Base,0.918400,0.002065,0.975669,NaN
K - + category_id,0.923286,0.001001,0.978289,0.004886
L - + fechas,0.925100,0.001729,0.979001,0.001814
M - + precio (redondez),0.926700,0.002062,0.979507,0.001600


**Conclusión.** Comparando cada ganancia contra el desvío entre folds (que ronda 0,001-0,0021 en estos bloques):

- **K (`category_id`, +0,0049)**: aporta de verdad — 2 a 5 veces el desvío. Menos que lo que sugería el EDA (categorías enteras con 100% `new` o 100% `used`), porque el `TargetEncoder` aplica *smoothing* justamente por la cantidad de categorías chicas, y porque parte de esa señal ya la capturaba el vendedor/título.
- **L (fechas, +0,0018)** y **M (precio/redondez, +0,0016)**: están al límite del ruido — la ganancia es del mismo orden que el desvío entre folds, no mucho más. No es que no sirvan: `tiene_decimales`/`termina_9` mostraban diferencias enormes en el EDA (hasta 46 puntos porcentuales), pero gran parte de esa señal probablemente **ya la captura el título** (un precio `$499,90` suele venir con palabras como "oferta" o nombres de marca que el TF-IDF ya ve) y el vendedor (una tienda que pone precios con decimales también tiene `seller_pct_new` alto). Es el caso típico de una variable con **correlación fuerte pero aporte marginal chico**, porque se solapa con señal que el modelo ya tenía.

En conjunto, de Base (0,9184) a acá (0,9267) son **+0,0083** de accuracy — bastante menos que el salto de B-E, título o vendedor, pero sigue siendo una mejora real, sobre todo por el bloque K.

**Recomendación para `train.ipynb`:** sumar `category_id` (bloque K, el aporte más sólido). Fechas y precio quedan para decidir — aportan algo pero son más marginales; si el tiempo apremia, se pueden dejar afuera sin perder mucho.